# 1. 데이터 로드
목적

Lending Club 원본 데이터를 불러오고, 분석에 사용할 기본 대상만 남깁니다.

코드 설명

loan_status가 확정된 대출만 사용하고, default라는 타겟 변수를 생성합니다.
또한 개인 대출(Individual)만 남겨서 분석 대상을 통일합니다.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import warnings

warnings.filterwarnings('ignore')

file_path = '/Users/hantaeho/Documents/statistic_science/stat_Assignment/lending_club_2020_train.csv'
df = pd.read_csv(file_path, low_memory=False)

valid_statuses = ['Fully Paid', 'Charged Off', 'Default']
df_base = df[df['loan_status'].isin(valid_statuses)].copy()

if 'application_type' in df_base.columns:
    df_base = df_base[df_base['application_type'] == 'Individual'].copy()

df_base['default'] = np.where(df_base['loan_status'].isin(['Charged Off', 'Default']), 1, 0)

print("데이터 형태:", df_base.shape)
df_base[['loan_status', 'default']].head()

데이터 형태: (1074490, 142)


,loan_status,default
2,Charged Off,1
5,Charged Off,1
6,Fully Paid,0
7,Fully Paid,0
8,Charged Off,1


# 2. 날짜 변환과 무위험수익률 준비
목적

샤프 레이시오 계산에 필요한 무위험수익률을 대출 발행 연도 기준으로 붙입니다.

코드 설명

issue_d, earliest_cr_line, last_pymnt_d를 날짜형으로 바꾸고,
발행 연도를 기준으로 국채 수익률을 매핑합니다.

In [2]:
for col in ['issue_d', 'earliest_cr_line', 'last_pymnt_d']:
    if col in df_base.columns:
        df_base[col] = pd.to_datetime(df_base[col], format='%b-%Y', errors='coerce')

df_base['issue_year'] = df_base['issue_d'].dt.year

treasury_rates = {
    2007: 0.044, 2008: 0.022, 2009: 0.014, 2010: 0.011,
    2011: 0.008, 2012: 0.004, 2013: 0.007, 2014: 0.015,
    2015: 0.013, 2016: 0.010, 2017: 0.016, 2018: 0.026,
    2019: 0.019, 2020: 0.004
}

df_base['risk_free_rate'] = df_base['issue_year'].map(treasury_rates).fillna(0.015)

df_base[['issue_d', 'earliest_cr_line', 'last_pymnt_d', 'issue_year', 'risk_free_rate']].head()

,issue_d,earliest_cr_line,last_pymnt_d,issue_year,risk_free_rate
2,2016-07-01,1993-05-01,2016-12-01,2016,0.010
5,2017-10-01,1984-05-01,2019-08-01,2017,0.016
6,2017-05-01,1998-07-01,2019-03-01,2017,0.016
7,2015-09-01,1999-11-01,2018-10-01,2015,0.013
8,2019-05-01,2002-03-01,2019-06-01,2019,0.019


# 3. 전처리
목적

모델 학습에 방해되는 컬럼을 제거하고, 수치형 입력으로 변환합니다.

코드 설명

사후 정보, 식별자, 텍스트성 컬럼을 제거하고,
term, emp_length, revol_util 같은 문자열형 변수는 숫자로 바꿉니다.
또한 비율형 파생변수와 결측치 지시자를 만들어 예측력을 보완합니다.

In [3]:
# 평가용 컬럼 먼저 저장
eval_cols = {}
for col in ['total_pymnt', 'funded_amnt', 'loan_status', 'default', 'issue_d', 'last_pymnt_d', 'earliest_cr_line', 'term']:
    if col in df_base.columns:
        eval_cols[col] = df_base[col].copy()
df_eval = pd.DataFrame(eval_cols, index=df_base.index)

# 사후 정보/식별자 제거
exclude_cols = [
    'int_rate', 'grade', 'sub_grade', 'installment',
    'funded_amnt', 'funded_amnt_inv', 'initial_list_status',
    'out_prncp', 'out_prncp_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee',
    'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d',
    'last_fico_range_low', 'last_fico_range_high',
    'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'hardship_start_date', 'hardship_end_date', 'hardship_amount',
    'hardship_length', 'hardship_dpd', 'hardship_loan_status',
    'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
    'deferral_term', 'payment_plan_start_date',
    'orig_projected_additional_accrued_interest',
    'debt_settlement_flag',
    'id', 'member_id', 'url', 'desc',
    'emp_title', 'title',
    'policy_code', 'pymnt_plan',
]
df_base.drop(columns=[c for c in exclude_cols if c in df_base.columns], inplace=True)

# term 숫자 변환
if 'term' in df_base.columns and df_base['term'].dtype == 'object':
    df_base['term'] = df_base['term'].str.extract(r'(\d+)').astype(float)

# 근속연수 숫자화
if 'emp_length' in df_base.columns and df_base['emp_length'].dtype == 'object':
    emp_map = {
        '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
        '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9, '10+ years': 10
    }
    df_base['emp_length'] = df_base['emp_length'].map(emp_map)

# revol_util 숫자화
if 'revol_util' in df_base.columns and df_base['revol_util'].dtype == 'object':
    df_base['revol_util'] = pd.to_numeric(
        df_base['revol_util'].astype(str).str.replace('%', '', regex=False).str.strip(),
        errors='coerce'
    )

# 결측치가 40% 이상인 열 제거
missing_ratio = df_base.isnull().sum() / len(df_base)
df_base = df_base[missing_ratio[missing_ratio < 0.4].index].copy()

# 파생변수
if 'fico_range_high' in df_base.columns and 'fico_range_low' in df_base.columns:
    df_base['fico_range_avg'] = (df_base['fico_range_high'] + df_base['fico_range_low']) / 2

if 'num_bc_sats' in df_base.columns and 'num_bc_tl' in df_base.columns:
    df_base['ratio_satis'] = (df_base['num_bc_sats'] / df_base['num_bc_tl']).replace([np.inf, -np.inf], np.nan).fillna(0)

if 'num_op_rev_tl' in df_base.columns and 'num_rev_accts' in df_base.columns:
    df_base['ratio_open_revolving'] = (df_base['num_op_rev_tl'] / df_base['num_rev_accts']).replace([np.inf, -np.inf], np.nan).fillna(0)

if 'issue_d' in df_base.columns and 'earliest_cr_line' in df_base.columns:
    df_base['credit_history_days'] = (df_base['issue_d'] - df_base['earliest_cr_line']).dt.days

# 결측치 지시자
missing_info_cols = ['revol_util', 'pct_tl_nvr_dlq', 'mths_since_recent_bc',
                     'percent_bc_gt_75', 'mo_sin_old_rev_tl_op', 'inq_last_6mths']

for col in missing_info_cols:
    if col in df_base.columns:
        df_base[col] = pd.to_numeric(df_base[col], errors='coerce')
        if df_base[col].isnull().sum() > 0:
            df_base[f'{col}_is_missing'] = np.where(df_base[col].isnull(), 1, 0)
            df_base[col] = df_base[col].fillna(df_base[col].mean())

# 범주형 처리
if 'zip_code' in df_base.columns:
    df_base.drop(columns=['zip_code'], inplace=True)

if 'addr_state' in df_base.columns:
    df_base.drop(columns=['addr_state'], inplace=True)

if 'purpose' in df_base.columns:
    df_base['purpose'] = df_base['purpose'].map(
        {'debt_consolidation': 'Financial', 'credit_card': 'Financial'}
    ).fillna('Other')

if 'home_ownership' in df_base.columns:
    df_base = df_base[df_base['home_ownership'].isin(['MORTGAGE', 'RENT', 'OWN'])]

if 'application_type' in df_base.columns:
    df_base.drop(columns=['application_type'], inplace=True)

dummy_cols = ['purpose', 'home_ownership', 'verification_status']
df_base = pd.get_dummies(df_base, columns=[c for c in dummy_cols if c in df_base.columns])

# 남은 문자열/날짜형 제거
obj_cols = df_base.select_dtypes(include=['object', 'datetime64']).columns.tolist()
if obj_cols:
    df_base.drop(columns=obj_cols, inplace=True)

# 수치형 결측치 평균 대치
numeric_cols = df_base.select_dtypes(include=[np.number]).columns
df_base[numeric_cols] = df_base[numeric_cols].fillna(df_base[numeric_cols].mean())

# 평가용 데이터 인덱스 맞추기
df_eval = df_eval.loc[df_base.index].copy()

print("전처리 후 형태:", df_base.shape)
print("남은 결측치 수:", df_base.isnull().sum().sum())

전처리 후 형태: (1073657, 74)
남은 결측치 수: 1073657


# 4. 학습/검증/테스트 분리
목적

모델 선택과 최종 평가를 분리해서 과적합을 줄입니다.

코드 설명

Train/Validation/Test를 60/20/20으로 나누고,
Train 셋에만 언더샘플링을 적용해 클래스 불균형을 완화합니다.

In [4]:
# 평가용 컬럼 다시 결합
df_base['total_pymnt'] = df_eval['total_pymnt']
df_base['default'] = df_eval['default']
df_base['issue_d'] = df_eval['issue_d']
df_base['last_pymnt_d'] = df_eval['last_pymnt_d']
df_base['term'] = df_eval['term']

# 상환기간
df_base['pymnt_term_years'] = (df_base['last_pymnt_d'] - df_base['issue_d']).dt.days / 365.25
df_base = df_base[df_base['pymnt_term_years'] > 0].copy()
df_eval = df_eval.loc[df_base.index].copy()

# 학습 변수 분리
leakage_cols = ['total_pymnt', 'last_pymnt_d', 'pymnt_term_years', 'loan_status', 'default', 'issue_d', 'earliest_cr_line']
features = [c for c in df_base.columns if c not in leakage_cols and df_base[c].dtype not in ['object', 'datetime64[ns]']]

X = df_base[features].copy()
y = df_base['default'].copy()

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp
)

# Train만 언더샘플링
train_data = pd.concat([X_train, y_train], axis=1)
df_majority = train_data[train_data['default'] == 0]
df_minority = train_data[train_data['default'] == 1]

df_majority_down = resample(
    df_majority,
    replace=False,
    n_samples=len(df_minority),
    random_state=42
)

train_downsampled = pd.concat([df_majority_down, df_minority]).sample(frac=1, random_state=42)

X_train_under = train_downsampled.drop('default', axis=1)
y_train_under = train_downsampled['default']

print(f'Train: {len(X_train):,}, Valid: {len(X_val):,}, Test: {len(X_test):,}')
print(f'언더샘플링 후 Train: 정상={len(df_majority_down):,}, 부도={len(df_minority):,}')

Train: 638,835, Valid: 212,945, Test: 212,945
언더샘플링 후 Train: 정상=123,068, 부도=123,068


# 5. 샤프 레이시오 계산 함수
목적

투자 성과를 평가할 기준을 정의합니다.

코드 설명

수익률에서 무위험수익률을 뺀 초과수익을 기준으로 샤프 레이시오를 계산합니다.
검증 샘플이 너무 적으면 안정성을 위해 NaN을 반환합니다.

In [5]:
# 숫자 변환
X_train_under = X_train_under.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
X_val = X_val.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)

treasury_3y = {
    2007: 4.35, 2008: 2.24, 2009: 1.43, 2010: 1.37, 2011: 0.75,
    2012: 0.36, 2013: 0.76, 2014: 0.97, 2015: 1.16, 2016: 1.01,
    2017: 1.62, 2018: 2.64, 2019: 1.62, 2020: 0.37,
}
treasury_5y = {
    2007: 4.43, 2008: 2.80, 2009: 2.20, 2010: 1.93, 2011: 1.52,
    2012: 0.76, 2013: 1.31, 2014: 1.64, 2015: 1.53, 2016: 1.32,
    2017: 1.92, 2018: 2.73, 2019: 1.69, 2020: 0.53,
}

def get_rf(issue_year, term_months):
    if pd.isna(issue_year) or pd.isna(term_months):
        return np.nan
    if int(term_months) == 60:
        return treasury_5y.get(int(issue_year), 1.5) / 100
    return treasury_3y.get(int(issue_year), 1.0) / 100

def calc_sharpe(returns, rf_rates):
    returns = np.asarray(returns, dtype=float)
    rf_rates = np.asarray(rf_rates, dtype=float)
    mask = np.isfinite(returns) & np.isfinite(rf_rates)
    returns = returns[mask]
    rf_rates = rf_rates[mask]
    if len(returns) < 30:
        return np.nan
    excess = returns - rf_rates
    std = np.std(excess, ddof=1)
    if std == 0 or np.isnan(std):
        return np.nan
    return np.mean(excess) / std

print("샤프 계산 함수 준비 완료")

샤프 계산 함수 준비 완료


# 6. 검증 셋 기준 baseline 평가
목적

모델을 쓰지 않고 전체 대출을 받는 경우와 비교할 기준선을 만듭니다.

코드 설명

검증 셋에서 대출별 수익률과 무위험수익률을 계산한 뒤,
전체 대출을 다 받는 전략의 샤프 레이시오를 구합니다.

In [6]:
val_eval = df_eval.loc[X_val.index].copy()
val_eval['issue_year'] = val_eval['issue_d'].dt.year
val_eval['term_months'] = pd.to_numeric(val_eval['term'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
val_eval['rf'] = val_eval.apply(lambda row: get_rf(row['issue_year'], row['term_months']), axis=1)

F_val = pd.to_numeric(val_eval['funded_amnt'], errors='coerce').values
P_val = pd.to_numeric(val_eval['total_pymnt'], errors='coerce').values
M_val = pd.to_numeric(val_eval['term_months'], errors='coerce').values
rf_val = val_eval['rf'].values

valid_ret_mask = np.isfinite(F_val) & np.isfinite(P_val) & np.isfinite(M_val) & (F_val > 0) & (M_val > 0)
r_val = np.full(len(F_val), np.nan, dtype=float)
r_val[valid_ret_mask] = (P_val[valid_ret_mask] / F_val[valid_ret_mask]) ** (12 / M_val[valid_ret_mask]) - 1

all_sharpe = calc_sharpe(r_val, rf_val)

print(f'전부 대출 시 샤프: {all_sharpe:.4f}')
print(f'국채 금리 범위: {np.nanmin(rf_val)*100:.2f}% ~ {np.nanmax(rf_val)*100:.2f}%')

전부 대출 시 샤프: -0.0967
국채 금리 범위: 0.36% ~ 4.35%


# 7. 모델 학습과 threshold 탐색
목적

부도 확률이 낮은 대출만 승인하도록 모델과 승인 기준을 함께 최적화합니다.

코드 설명

여러 분류기를 학습한 뒤, 검증 셋에서 threshold를 바꿔가며
샤프 레이시오가 가장 큰 지점을 찾습니다.

In [7]:
models_7 = {
    'LightGBM': LGBMClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1, verbose=-1),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1, eval_metric='logloss', verbosity=0),
    'CatBoost': CatBoostClassifier(iterations=100, depth=5, random_state=42, verbose=0),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1),
    'Grad Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Naive Bayes': GaussianNB()
}

best_models = {}

for name, model in models_7.items():
    model.fit(X_train_under, y_train_under)
    proba_val = model.predict_proba(X_val)[:, 1]

    best_t, best_sharpe = 0.5, -np.inf
    grid = np.quantile(proba_val, np.linspace(0.05, 0.95, 60))

    for t in grid:
        mask = proba_val <= t
        if mask.sum() < 500:
            continue
        sh = calc_sharpe(r_val[mask], rf_val[mask])
        if np.isfinite(sh) and sh > best_sharpe:
            best_sharpe = sh
            best_t = t

    approved = proba_val <= best_t
    best_models[name] = {'model': model, 'threshold': best_t, 'sharpe': best_sharpe}

    print(f'[{name:15s}] 샤프: {best_sharpe:7.4f} | threshold: {best_t:.3f} | 승인: {approved.sum():,}건 ({approved.mean()*100:.1f}%) | vs baseline: {best_sharpe - all_sharpe:+.4f}')

winner = max(best_models, key=lambda k: best_models[k]['sharpe'])
print(f'\n최우수 모델: {winner} | 샤프: {best_models[winner]["sharpe"]:.4f}')

[LightGBM       ] 샤프:  1.2914 | threshold: 0.183 | 승인: 153,573건 (72.1%) | vs baseline: +1.3881
[XGBoost        ] 샤프:  1.2793 | threshold: 0.167 | 승인: 153,573건 (72.1%) | vs baseline: +1.3760
[CatBoost       ] 샤프:  1.2873 | threshold: 0.157 | 승인: 150,325건 (70.6%) | vs baseline: +1.3841
[Random Forest  ] 샤프:  0.5781 | threshold: 0.303 | 승인: 13,896건 (6.5%) | vs baseline: +0.6748
[Grad Boosting  ] 샤프:  1.2934 | threshold: 0.188 | 승인: 153,573건 (72.1%) | vs baseline: +1.3901
[AdaBoost       ] 샤프:  1.2346 | threshold: 0.467 | 승인: 117,842건 (55.3%) | vs baseline: +1.3313
[Naive Bayes    ] 샤프:  0.0924 | threshold: 0.008 | 승인: 36,634건 (17.2%) | vs baseline: +0.1891

최우수 모델: Grad Boosting | 샤프: 1.2934


# 8. Test 셋 최종 평가
목적

검증 셋에서 고른 threshold가 실제로도 좋은지 확인합니다.

코드 설명

선택된 최우수 모델과 threshold를 Test 셋에 적용하고,
최종 샤프 레이시오와 승인 비율을 출력합니다.

In [8]:
best_model = best_models[winner]['model']
best_t = best_models[winner]['threshold']

test_eval = df_eval.loc[X_test.index].copy()
test_eval['issue_year'] = test_eval['issue_d'].dt.year
test_eval['term_months'] = pd.to_numeric(test_eval['term'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
test_eval['rf'] = test_eval.apply(lambda row: get_rf(row['issue_year'], row['term_months']), axis=1)

F_test = pd.to_numeric(test_eval['funded_amnt'], errors='coerce').values
P_test = pd.to_numeric(test_eval['total_pymnt'], errors='coerce').values
M_test = pd.to_numeric(test_eval['term_months'], errors='coerce').values
rf_test = test_eval['rf'].values

valid_test_mask = np.isfinite(F_test) & np.isfinite(P_test) & np.isfinite(M_test) & (F_test > 0) & (M_test > 0)
r_test = np.full(len(F_test), np.nan, dtype=float)
r_test[valid_test_mask] = (P_test[valid_test_mask] / F_test[valid_test_mask]) ** (12 / M_test[valid_test_mask]) - 1

proba_test = best_model.predict_proba(X_test)[:, 1]
approved_test = proba_test <= best_t

test_sharpe = calc_sharpe(r_test[approved_test], rf_test[approved_test])
baseline_test = calc_sharpe(r_test, rf_test)

print('=== Test 결과 ===')
print(f'Baseline Sharpe (전체 대출): {baseline_test:.4f}')
print(f'Winner Sharpe (threshold 적용): {test_sharpe:.4f}')
print(f'승인 비율: {approved_test.mean()*100:.1f}%')

=== Test 결과 ===
Baseline Sharpe (전체 대출): -0.0958
Winner Sharpe (threshold 적용): 1.2926
승인 비율: 72.2%


In [12]:
X_train_under.head()

,loan_amnt,annual_inc,dti,delinq_2yrs,fico_range_low,fico_range_high,inq_last_6mths,open_acc,pub_rec,revol_bal,...,mo_sin_old_rev_tl_op_is_missing,purpose_Financial,purpose_Other,home_ownership_MORTGAGE,home_ownership_OWN,home_ownership_RENT,verification_status_Not Verified,verification_status_Source Verified,verification_status_Verified,term
1145653,21000.0,50000.0,30.84,1.0,670.0,674.0,0.0,16.0,0.0,18006.0,...,0,True,False,True,False,False,True,False,False,0.0
1465,20000.0,190000.0,32.37,0.0,700.0,704.0,0.0,17.0,0.0,43859.0,...,0,True,False,True,False,False,True,False,False,0.0
286102,2000.0,49747.0,22.65,1.0,675.0,679.0,0.0,9.0,0.0,2518.0,...,0,False,True,False,False,True,False,False,True,0.0
435432,15000.0,60000.0,21.44,0.0,710.0,714.0,1.0,17.0,0.0,20642.0,...,0,False,True,True,False,False,True,False,False,0.0
623582,26600.0,70000.0,14.35,0.0,665.0,669.0,1.0,10.0,0.0,10879.0,...,0,True,False,True,False,False,False,False,True,0.0
